# 02 · Which image model and classifier?

Only the 66 competition photos are used here (no external data), so this is the starting point.
Each of the 5 embeddings (and all five joined together) is tried with four classifiers:

- **logistic regression** (C = 1)
- **RBF SVM** — support vector machine with a radial basis function kernel (C = 3)
- **extra trees** — 300 randomised decision trees
- **neural net (MLP)** — one hidden layer of 256 units, trained with PyTorch (see `common.TorchMLP`)

All use class-balanced weights. Host baselines on the public leaderboard: trained 0.747, random 0.326.

In [1]:
import numpy as np, pandas as pd
import common as c
pd.set_option('display.width', 200)

In [2]:
df = c.training_set()                      # the 66 competition photos only
feats = {b: c.embed(df.id, b) for b in c.BACKBONES}
feats["all 5 joined"] = c.unit(np.hstack(list(feats.values())))
feats["dinov2_l (photo + mirror + 5 crops)"] = c.embed(df.id, "dinov2_l", view="mean7")
res = pd.concat([c.cross_validate(df, X, list(c.MODELS), name=n)[0] for n, X in feats.items()])
res.pivot_table(index="experiment", columns="model", values="CV macro F1").round(3)

model,RBF SVM,extra trees,logistic regression,neural net (MLP)
experiment,,,,
all 5 joined,0.810,0.826,0.814,0.795
dinov2_l,0.800,0.819,0.808,0.776
dinov2_l (photo + mirror + 5 crops),0.795,0.795,0.788,0.773
eva02_l,0.768,0.795,0.778,0.754
siglip_so400m,0.776,0.800,0.785,0.808
vith14_laion,0.756,0.723,0.730,0.749
vitl14_openai,0.709,0.680,0.709,0.698


In [3]:
best = res.sort_values("CV macro F1", ascending=False).head(5)
best

,experiment,model,CV macro F1,std,dishes F1,trash F1
2,all 5 joined,extra trees,0.826,0.036,0.820,0.832
2,dinov2_l,extra trees,0.819,0.024,0.824,0.816
0,all 5 joined,logistic regression,0.814,0.019,0.816,0.813
1,all 5 joined,RBF SVM,0.810,0.023,0.797,0.824
3,siglip_so400m,neural net (MLP),0.808,0.028,0.863,0.756


**Result.** With 66 photos the image models score between about 0.68 (OpenAI CLIP) and 0.83 (all five joined, extra
trees). DINOv2 is among the best single models (0.80–0.82) and SigLIP is close; joining all five adds little for the
extra cost. Differences of 0.02 are within the noise of 10 repeats. Averaging the photo, mirror and five crops
(multi-crop feature averaging) does not help. DINOv2 is the image model used from here on, because its features also
work for the zoomed-in tiles used by the helpers in notebook 05. All four classifiers are kept side by side in later
notebooks.